# Generative AI 016 — RAG: Concept, History and Architecture

Every part of RAG was built in lessons 005–015, so here the **whole four-step
pipeline runs end to end with no API key**. The model is a stub that echoes
back its prompt — so you see exactly what a real model would be sent.

| Part | What we check |
|---|---|
| A | the full pipeline, and the exact prompt it produces |
| B | how often retrieval finds the right passage: **6/6** vs **1/4** at k=2 |
| C | adding knowledge without retraining — one call, about a millisecond |

The history in the lesson — fine-tuning, in-context learning emerging at
GPT-3 scale — is **reported from the literature**, not measured here.

Needs `langchain-core`, `scikit-learn`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Indexing, retrieval, augmentation, generation

The four steps are labelled in the code below.

In [ ]:
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEmbeddings(Embeddings):
    """Offline stand-in for an embedding model. Compares WORDS, not meaning."""
    def __init__(self, corpus):
        self.v = TfidfVectorizer(stop_words="english").fit(corpus)
    def embed_documents(self, texts):
        return self.v.transform(texts).toarray().tolist()
    def embed_query(self, text):
        return self.v.transform([text]).toarray()[0].tolist()

LECTURE = [   # a linear-regression lecture, as timed transcript segments
    ("00-05", "Welcome. Today we study linear regression, which fits a straight "
              "line to data by choosing a slope and an intercept."),
    ("05-15", "Gradient descent is the optimisation step. We start with random "
              "weights, compute the gradient of the loss, and move the weights a "
              "small step in the opposite direction, repeating until the loss "
              "stops falling."),
    ("15-25", "The learning rate controls the size of each gradient descent step. "
              "Too large and the loss diverges; too small and training crawls."),
    ("25-40", "The loss function for regression is mean squared error, the "
              "average of the squared differences between predictions and targets."),
    ("40-55", "Overfitting happens when the model memorises noise. Regularisation "
              "such as ridge adds a penalty on large weights to prevent it."),
    ("55-70", "We evaluate the model with R squared, which measures the fraction "
              "of variance in the target that the model explains."),
    ("70-90", "Feature scaling puts inputs on similar ranges, which makes gradient "
              "descent converge faster and more reliably."),
    ("90-120", "Finally, the normal equation solves linear regression in closed "
               "form without any iteration, but it becomes slow for many features."),
]

# ---- 1. INDEXING: load, (already split), embed, store ------------------
store = InMemoryVectorStore(embedding=TfidfEmbeddings([t for _, t in LECTURE]))
store.add_documents([Document(page_content=t, metadata={"minutes": m})
                     for m, t in LECTURE])

# ---- 2. RETRIEVAL -------------------------------------------------------
retriever = store.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(f"[minutes {d.metadata['minutes']}] {d.page_content}"
                       for d in docs)

# ---- 3. AUGMENTATION ----------------------------------------------------
prompt = PromptTemplate.from_template(
    "You are a helpful teaching assistant. Answer ONLY from the provided "
    "context. If the context is insufficient, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}")

# ---- 4. GENERATION - a stub that ECHOES its prompt, so we can read it ---
class EchoModel(FakeListChatModel):
    def _call(self, messages, stop=None, run_manager=None, **kwargs):
        return messages[-1].content

chain = ({"context": retriever | RunnableLambda(format_docs),
          "question": RunnablePassthrough()}
         | prompt | EchoModel(responses=["unused"]) | StrOutputParser())

print(chain.invoke("How do we perform the optimisation step in gradient descent?"))

# ...Context:
# [minutes 05-15] Gradient descent is the optimisation step. We start with ...
# [minutes 15-25] The learning rate controls the size of each gradient ...
# Question: How do we perform the optimisation step in gradient descent?
#
# That is everything a real model would be sent: the question, the two
# segments the retriever chose WITH their timestamps, and an instruction to
# answer only from them. Swap EchoModel for ChatOpenAI() and it is a working
# RAG system. The chain is lessons 009-015 in one expression.

In [ ]:
sent = chain.invoke("How do we perform the optimisation step in gradient descent?")
assert "[minutes 05-15]" in sent            # the right segment was retrieved
assert "Answer ONLY from the provided context" in sent
print(len(sent), "characters sent;", sum(len(t) for _, t in LECTURE), "in the whole lecture")

The prompt holds **k segments however long the lecture is** — grow the lecture
a hundredfold and the prompt stays the same size. That is why RAG scales.

## Part B — Retrieval decides everything

In [ ]:
QUESTIONS = [   # (question, the segment that answers it, phrasing)
    ("How does gradient descent update the weights?", "05-15", "direct"),
    ("What does the learning rate control?", "15-25", "direct"),
    ("What is mean squared error?", "25-40", "direct"),
    ("How does ridge regularisation stop overfitting?", "40-55", "direct"),
    ("What does R squared measure?", "55-70", "direct"),
    ("Why should I scale my features?", "70-90", "direct"),
    ("How do we tweak the parameters to reduce the error?", "05-15", "paraphrase"),
    ("How big should each step be?", "15-25", "paraphrase"),
    ("How do we stop the model memorising the training set?", "40-55", "paraphrase"),
    ("Can we solve it without iterating?", "90-120", "paraphrase"),
]

def hit_rate(phrasing, k):
    qs = [q for q in QUESTIONS if q[2] == phrasing]
    got = 0
    for q, want, _ in qs:
        try:
            found = [d.metadata["minutes"] for d in store.similarity_search(q, k=k)]
        except ValueError:          # a query with no known words -> zero vector
            found = []
        got += want in found
    return f"{got}/{len(qs)}"

for phrasing in ("direct", "paraphrase"):
    print(phrasing, [hit_rate(phrasing, k) for k in (1, 2, 3)])

# direct     ['4/6', '6/6', '6/6']
# paraphrase ['0/4', '1/4', '2/4']
#
# Questions in the lecture's own words are answered. The same questions in a
# student's words mostly are not - and no prompt instruction can rescue an
# answer that was never retrieved. A RAG system is only as good as its
# retrieval. (TF-IDF exaggerates the gap; the direction is real.)

In [ ]:
assert hit_rate("direct", 2) == "6/6"
assert hit_rate("paraphrase", 2) == "1/4"

# Two 'direct' misses at k=1 are pure text-processing:
vec = TfidfVectorizer(stop_words="english").fit([t for _, t in LECTURE])
print(vec.build_analyzer()("What does R squared measure?"))    # the letter R is dropped
print(vec.build_analyzer()("Why should I scale my features?")) # 'scale' is not 'scaling'

**A RAG system is only as good as its retrieval.** The model is told to answer
only from the context — so a wrong passage means a wrong answer, however good
the prompt.

> TF-IDF compares words, not meaning, so the paraphrase row is much worse than
> a real embedding model would manage. The *direction* is real.

## Part C — New knowledge, no retraining

In [ ]:
import time

question = "What did the guest lecture say about logistic regression?"

new_doc = Document(
    page_content="In the guest lecture on logistic regression, the model predicts "
                 "a probability by passing a linear score through the sigmoid.",
    metadata={"minutes": "guest"})

t0 = time.perf_counter()
store.add_documents([new_doc])
print(f"added in {(time.perf_counter() - t0) * 1000:.2f} ms")

print(store.similarity_search(question, k=1)[0].metadata)   # {'minutes': 'guest'}

# New knowledge is a write to a database, not a training run - that is what
# "no retraining" means, made concrete. What fine-tuning the same fact in
# would cost is NOT measured here: it needs a GPU, a dataset and a training
# run, and quoting a figure without them would be inventing one.

In [ ]:
assert store.similarity_search(question, k=1)[0].metadata["minutes"] == "guest"
print("the new document is retrievable immediately")

## What to take away

- **Parametric knowledge** fails on private data, recent data and hallucination.
- **RAG** puts the *knowledge* in the prompt, not examples of how to answer.
- **Four steps**: indexing, retrieval, augmentation, generation — all built
  earlier in the course.
- Retrieval: **6/6** at k=2 in the lecture's words, **1/4** in a student's.
- New knowledge is **one write**, about a millisecond — no retraining.

## Exercises

1. Raise k to 4. Does the paraphrase hit rate reach 4/4? What does the prompt
   cost you in characters?
2. Write three more paraphrased questions a real student might ask. Predict
   before running whether each will be retrieved.
3. Add stemming (lowercase + strip a trailing "ing"/"s") to the embedder. How
   much of the gap does that close — and why can it never close all of it?
4. Remove "Answer ONLY from the provided context" from the prompt. With a real
   model and a retrieval miss, what would you expect to change?
5. If you have an API key, swap `EchoModel` for a real chat model and ask the
   four paraphrased questions. Does the model say "I don't know" when retrieval
   missed?